In [ ]:
!pip install pydub
!pip install whisperx

In [ ]:
# from pydub import AudioSegment
# from pydub.silence import detect_silence

# audio = AudioSegment.from_file("006008.wav")

# # silence threshold in milliseconds
# silences = detect_silence(
#     audio,
#     min_silence_len=500,   # minimum silent duration (ms)
#     silence_thresh=-40     # volume threshold (dBFS)
# )

# for start, end in silences:
#     print(
#         f"Silence from {start/1000:.2f}s to {end/1000:.2f}s"
#     )

Silence from 6.59s to 8.14s
Silence from 17.23s to 18.23s


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import kagglehub
path = kagglehub.dataset_download("omartariq612/quran-reciters")

100%|██████████| 11.0G/11.0G [02:06<00:00, 93.9MB/s]

Extracting files...


In [ ]:
import whisperx
from pydub import AudioSegment
from pydub.silence import detect_silence

def get_word_timestamps(audio_path, device="cpu"):
    model = whisperx.load_model("large-v2", device=device, language="ar")
    result = model.transcribe(audio_path, language="ar")

    align_model, metadata = whisperx.load_align_model(language_code="ar", device=device)
    result = whisperx.align(result["segments"], align_model, metadata, audio_path, device)

    words = []
    for seg in result["segments"]:
        for w in seg.get("words", []):
            words.append({
                "word": w["word"],
                "start": w["start"],  # seconds
                "end":   w["end"]
            })
    return words


def get_silence_intervals(audio_path, min_silence_len=300, silence_thresh=-40):
    audio = AudioSegment.from_file(audio_path)
    silences = detect_silence(audio, min_silence_len=min_silence_len, silence_thresh=silence_thresh)
    # convert ms → seconds
    return [(s / 1000, e / 1000) for s, e in silences]


def intervals_overlap(a_start, a_end, b_start, b_end, min_overlap=0.1):
    overlap = min(a_end, b_end) - max(a_start, b_start)
    return overlap >= min_overlap


def inject_sil_tokens(phoneme_sequence, words, silence_intervals, min_overlap=0.1):
    """
    phoneme_sequence : list like ['l','aa','<space>','t','u',...]
    words            : list of {word, start, end} from WhisperX
    silence_intervals: list of (start_sec, end_sec) from pydub
    """



    # Find indices of all <space> tokens
    space_indices = [i for i, p in enumerate(phoneme_sequence) if p == "<space>"]

    # Guard: phoneme word count must match WhisperX word count
    if len(space_indices) != len(words) - 1:
        return phoneme_sequence

    # Each <space> sits between word[k] and word[k+1]
    result = phoneme_sequence.copy()

    for k, space_idx in enumerate(space_indices):
        if k + 1 >= len(words):
            break

        gap_start = words[k]["end"]
        gap_end   = words[k + 1]["start"]

        # Check if any silence interval overlaps this gap
        for sil_start, sil_end in silence_intervals:
            if intervals_overlap(gap_start, gap_end, sil_start, sil_end, min_overlap):
                result[space_idx] = "<sil>"
                break  # one match is enough

    return result

In [ ]:
# --- Main ---
audio_path = "006008.wav"

phonemes = ['l','aa','<space>','t','u','d','r','i','k','u','h','u',
            '<space>','<wasl>','a','l','ʔ','a','b','sˤ','aa','r','u']

words            = get_word_timestamps(audio_path)
silence_intervals = get_silence_intervals(audio_path, min_silence_len=300, silence_thresh=-40)

updated_phonemes = inject_sil_tokens(phonemes, words, silence_intervals)
print(updated_phonemes)